In [1]:
%run_nb spark-start

Args: Namespace(data_format='none', port_offset=2) - unknown_args: []
Spark version: 4.1.3, Driver memory: 16g, Executor memory: 8g, Service: jupyter-spark-4.1, Data format: None
Spark packages: org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.3
Spark extensions: 
Spark catalog configs: {}
spark.sql.shuffle.partitions: 200
spark.sparkContext.master: local[2]


Loading ITables v2.9.1 from the internet... (need help?)
🔒ⓘcatalog
spark_catalog


Version,4.1.3
Master,local[2]
AppName,main


               total        used        free      shared  buff/cache   available
Mem:            62Gi        17Gi        27Gi       461Mi        18Gi        45Gi
Swap:          8.0Gi          0B       8.0Gi


In [2]:
!kafkactl get topics
!kafkactl get brokers

TOPIC        PARTITIONS     REPLICATION FACTOR
my-topic     1              1
ID     ADDRESS
1      kafka-4-backend:29092


In [3]:
!kafkactl delete topic my-topic

topic deleted: my-topic


In [4]:
%%bash 
kafkactl create topic my-topic \
  --partitions 1 \
  --replication-factor 1

topic created: my-topic


In [5]:
!nc -vz kafka-4-backend 29092

Connection to kafka-4-backend (172.18.0.6) 29092 port [tcp/*] succeeded!


In [6]:
import os

from pyspark.sql import SparkSession


bootstrap_servers = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka-4-backend:29092")
topic = os.getenv("KAFKA_TOPIC", "my-topic")
truststore_location = os.getenv(
    "KAFKA_TRUSTSTORE_LOCATION",
    "/etc/kafka/tls/kafka.truststore.p12",
)
truststore_password = os.getenv("KAFKA_TRUSTSTORE_PASSWORD", "changeit")

In [7]:
messages = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", topic)
    .option("startingOffsets", "earliest")
    .option("kafka.security.protocol", "SSL")
    .option("kafka.ssl.truststore.location", truststore_location)
    .option("kafka.ssl.truststore.password", truststore_password)
    .option("kafka.ssl.truststore.type", "PKCS12")
    .load()
    .selectExpr(
        "CAST(key AS STRING) AS key",
        "CAST(value AS STRING) AS value",
        "topic",
        "partition",
        "offset",
        "timestamp",
    )
)

In [8]:
query = (
    messages.writeStream.format("console")
    .outputMode("append")
    .option("truncate", "false")
    .start()
)

try:
    query.awaitTermination(10) # Wait 10 seconds
finally:
    query.stop()